In [1]:
# general libraries
import numpy as np
import matplotlib.pyplot as plt

# main libraries
import keras
import tensorflow as tf
from keras.layers import Conv2D, MaxPool2D, Dropout, Flatten, Dense, BatchNormalization, GlobalAvgPool2D
from keras.models import Sequential
from keras.callbacks import ModelCheckpoint, EarlyStopping

from tensorflow.keras.preprocessing.image import ImageDataGenerator


2026-02-04 21:11:31.194616: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-04 21:11:31.195018: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-04 21:11:31.196890: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-04 21:11:31.202672: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-04 21:11:31.214100: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registe

In [2]:
model = Sequential()
model.add(Conv2D(filters = 16, kernel_size = (3,3), activation="relu", input_shape=(224,224,3) ))

model.add(Conv2D(filters = 32, kernel_size = (3,3), activation="relu" ))
model.add(MaxPool2D(pool_size=(2,2)))

model.add(Conv2D(filters = 64, kernel_size = (3,3), activation="relu" ))
model.add(MaxPool2D(pool_size=(2,2)))

model.add(Conv2D(filters = 128, kernel_size = (3,3), activation="relu" ))
model.add(MaxPool2D(pool_size=(2,2)))

model.add(Dropout(rate = .25))

model.add(Flatten())
model.add(Dense(units = 64, activation= "relu"))
model.add(Dropout(rate = 0.25))
model.add(Dense( units = 1, activation= "sigmoid"))

model.compile(optimizer="adam",loss=keras.losses.BinaryCrossentropy(),metrics=["accuracy"])

model.summary()



/home/coco/Learning/personal/Brain-Tumor-Detection-Model-using-Keras/venv/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-02-04 21:11:37.128941: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-02-04 21:11:37.131380: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://ww

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 220, 220, 32)   │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 110, 110, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 108, 108, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     5,537,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,635,361 (21.50 MB)

 Trainable params: 5,635,361 (21.50 MB)

 Non-trainable params: 0 (0.00 B)

In [3]:
def imagePreaparation1(path):

    # ip : Path
    # op : processed images

    imageData = ImageDataGenerator(zoom_range = .2,         #data augmentation
                                   shear_range = .2,
                                   rescale = 1/255,
                                   horizontal_flip = True)

    image = imageData.flow_from_directory(directory = path,
                                          target_size = (224,224),
                                          batch_size = 32,
                                          class_mode = "binary")

    return image


In [4]:
def imagePreaparation2(path):

    # ip : Path
    # op : processed images

    imageData = ImageDataGenerator(rescale = 1/255)

    image = imageData.flow_from_directory(directory = path,
                                          target_size = (224,224),
                                          batch_size = 32,
                                          class_mode = "binary")

    return image


In [5]:
# calling and preaparing
tainingDataPath = "data/Training"
trainingData = imagePreaparation1(tainingDataPath)
testingDataPath = "data/Testing"
testingData = imagePreaparation2(testingDataPath)
validationDataPath = "data/Validation"
validationData = imagePreaparation2(validationDataPath)



Found 5712 images belonging to 2 classes.
Found 666 images belonging to 2 classes.
Found 645 images belonging to 2 classes.


In [6]:
# early stopping and model checking
# early stopping
es = EarlyStopping(monitor="val_accuracy",
                   min_delta=.01,
                   patience=3,
                   verbose=1,
                   mode= 'auto')


In [7]:
# model check point
mc = ModelCheckpoint(monitor="val_accuracy",
                     filepath='./bestModel.keras',
                     verbose=1,
                     save_best_only=True,
                     mode='auto')

cd = [es,mc]


In [8]:
# Model Training
hs = model.fit(trainingData,
               steps_per_epoch= 8,
               epochs = 30,
               verbose = 1,
               validation_data = validationData,
               validation_steps = 16,
               callbacks = cd)




Epoch 1/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 522ms/step - accuracy: 0.6009 - loss: 0.7824
Epoch 1: val_accuracy improved from None to 0.69727, saving model to ./bestModel.keras

Epoch 1: finished saving model to ./bestModel.keras
8/8 ━━━━━━━━━━━━━━━━━━━━ 8s 836ms/step - accuracy: 0.6289 - loss: 0.7269 - val_accuracy: 0.6973 - val_loss: 0.6434
Epoch 2/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 553ms/step - accuracy: 0.6811 - loss: 0.6702
Epoch 2: val_accuracy did not improve from 0.69727
8/8 ━━━━━━━━━━━━━━━━━━━━ 6s 825ms/step - accuracy: 0.6953 - loss: 0.6333 - val_accuracy: 0.6855 - val_loss: 0.6020
Epoch 3/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 524ms/step - accuracy: 0.7458 - loss: 0.5678
Epoch 3: val_accuracy improved from 0.69727 to 0.76172, saving model to ./bestModel.keras

Epoch 3: finished saving model to ./bestModel.keras
8/8 ━━━━━━━━━━━━━━━━━━━━ 6s 816ms/step - accuracy: 0.7812 - loss: 0.5408 - val_accuracy: 0.7617 - val_loss: 0.4908
Epoch 4/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 552ms/step - accuracy: 0.8379 